# Session 04: Practice Quiz

**Notebook 2 of 3: Self-check** · about 35 minutes

Do this after `01-concepts.ipynb` and `03-debugging-lab.ipynb`. For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

## Q1 · Predict the output

`settings.py`:

```python
print("settings loaded")
TIMEOUT_SECONDS = 30
```

`main.py`, run with `python main.py`:

```python
import settings
from settings import TIMEOUT_SECONDS
import settings

settings.TIMEOUT_SECONDS = 5
print(TIMEOUT_SECONDS, settings.TIMEOUT_SECONDS)
print(__name__, settings.__name__)
```

<details>
<summary>Show answer</summary>

```text
settings loaded
30 5
__main__ settings
```

- The module's code runs **once**; the later imports reuse the cached module from `sys.modules`.
- `from settings import TIMEOUT_SECONDS` binds a new name in `main` to the object `30` *at import time*. Rebinding `settings.TIMEOUT_SECONDS` later changes the module's attribute, not `main`'s name. It's names and objects again (Session 02): two names, and only one of them was rebound.
- The script you run is always `__main__`; imported modules get their own names.

**Practical consequence:** if code needs to see changes to a module-level setting, read it as `settings.TIMEOUT_SECONDS` at the point of use. Better still, pass configuration in explicitly, as `incident_report` does with `Config`.

</details>

## Q2 · Debugging: it works with `uv run`, but not from VS Code's play button

A teammate's project uses the `src/` layout. In the terminal, `uv run incident-report` works. In VS Code, they open `src/incident_report/cli.py` and press **Run Python File**, and get:

```text
ModuleNotFoundError: No module named 'incident_report'
```

Give two possible causes, and the fix for each.

<details>
<summary>Show answer</summary>

1. **VS Code is using a different interpreter**, one that doesn't have the package installed: Session 01's "wrong environment" bug. Fix: **Python: Select Interpreter**, then choose the project's `.venv`.
2. **The right `.venv`, but the package was never installed into it.** For example, the environment was created by hand with `python -m venv`, or by a tool that installed only the dependencies. Fix: run `uv sync`, which installs `incident_report` itself in editable mode.

Even once the import works, **Run Python File** is the wrong way to start this program. `cli.py` only *defines* `main()`, so run as a file it does nothing at all. And if the package used relative imports, running it as a file would fail. Run packages the way users do: `uv run incident-report`, `uv run python -m incident_report`, or a VS Code debug configuration with `"module": "incident_report"`.

</details>

## Q3 · Predict the output: dataclasses

In [ ]:
from dataclasses import FrozenInstanceError, dataclass, replace


@dataclass(frozen=True)
class Rejection:
    file: str
    record: int | None
    problems: tuple[str, ...] = ()


first = Rejection("a.json", 3, ("missing field 'id'",))
second = Rejection("a.json", 3, ("missing field 'id'",))

print(first == second, first is second)
print(first)
try:
    first.record = 4
except FrozenInstanceError:
    print("cannot change a frozen dataclass")
print(replace(first, record=4).record, first.record)
print(len({first, second}))

<details>
<summary>Show answer</summary>

```text
True False
Rejection(file='a.json', record=3, problems=("missing field 'id'",))
cannot change a frozen dataclass
4 3
1
```

- The generated `__eq__` compares field values, so they're equal, but they're two objects.
- The generated `__repr__` shows every field.
- `frozen=True` blocks assignment; `replace()` builds a new instance instead.
- Frozen dataclasses with hashable fields are **hashable**, and equal instances hash the same, so a set keeps one of them. That's why `problems` is a tuple: with a list field, `hash()` would fail.

</details>

## Q4 · Debugging: the class that won't define

```python
from dataclasses import dataclass

@dataclass
class ExportBatch:
    folder: str
    files: list[str] = []
    errors: dict[str, str] = {}
```

Running the module fails before anything else happens. What's the error, why does Python refuse, and what's the fix?

<details>
<summary>Show answer</summary>

`ValueError: mutable default <class 'list'> for field files is not allowed: use default_factory`.

A default value is created **once**, when the class is defined, so every `ExportBatch` would share the same list and dict: Session 03's mutable-default trap. Dataclasses detect this for `list`, `dict` and `set`, and refuse.

```python
from dataclasses import dataclass, field

@dataclass
class ExportBatch:
    folder: str
    files: list[str] = field(default_factory=list)
    errors: dict[str, str] = field(default_factory=dict)
```

`default_factory` is called for each new instance.

</details>

## Q5 · Engineering judgment: inheritance or composition?

For each design, would you use inheritance, composition, or neither?

1. `IncidentReportError(Exception)`, with `ConfigError` and `ExportError` as subclasses.
2. `class JsonReport(Report)`, `class MarkdownReport(Report)`, with a base `Report` that holds the data and subclasses that only differ in `render()`.
3. `class TicketWithSla(Ticket)`, adding an `sla_hours` field so tickets can check their own breaches.
4. A `NightlyJob` that needs a config, a loader and a writer.

<details>
<summary>Show answer</summary>

1. **Inheritance.** Exceptions are the textbook case: callers can catch the whole family with `except IncidentReportError`, or one member specifically. One level, no shared state.
2. **Probably neither.** A report is data, so one dataclass, plus a separate renderer for each format: `render_json(report)` and `render_markdown(report)`, perhaps chosen through a dict of functions (Session 03). Adding a format then never touches the data class.
3. **Composition**, or just a parameter. The SLA table is configuration, not a property of a ticket. `ticket.is_breached(sla_hours)` keeps `Ticket` simple and lets the same ticket be checked against different policies.
4. **Composition.** `NightlyJob` *has* a config, *uses* a loader and a writer. In tests you can pass in a fake loader, which is far easier than overriding methods of a base class.

</details>

## Q6 · Implementation: a parametrized test

Write **one** parametrized test for `priority_rank`, covering: `"P1"` → `1`, `"p3"` → `3`, `" P2 "` → `2`, and `"P9"` raising `ValueError`. The cell writes your test to a temporary folder and runs pytest on it.

In [ ]:
import os
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

CODE = """
def priority_rank(priority: str) -> int:
    cleaned = priority.strip().upper()
    if cleaned not in {"P1", "P2", "P3", "P4"}:
        raise ValueError(f"unknown priority {priority!r}")
    return int(cleaned[1])
"""

MY_TESTS = """
import pytest

from ranks import priority_rank

# Write your parametrized test (and the ValueError test) here.
"""

with tempfile.TemporaryDirectory() as workdir:
    Path(workdir, "ranks.py").write_text(textwrap.dedent(CODE), encoding="utf-8")
    Path(workdir, "test_ranks.py").write_text(textwrap.dedent(MY_TESTS), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
        cwd=workdir, capture_output=True, text=True, encoding="utf-8",
        env=os.environ | {"PY_COLORS": "0"},  # plain text output
    )
    print(result.stdout[-600:])

<details>
<summary>Show answer</summary>

```python
import pytest

from ranks import priority_rank


@pytest.mark.parametrize(
    ("priority", "expected"),
    [("P1", 1), ("p3", 3), (" P2 ", 2)],
)
def test_priority_rank(priority, expected):
    assert priority_rank(priority) == expected


def test_unknown_priority_is_rejected():
    with pytest.raises(ValueError, match="P9"):
        priority_rank("P9")
```

Expected: `4 passed`.

The error case gets its own test rather than a parametrize row. Mixing "returns a value" and "raises" in one parametrized test needs awkward branching inside the test, and a test should do one thing. `match=` checks the message too, so a `ValueError` raised for some *other* reason doesn't count as a pass.

</details>

## Q7 · Predict: fixtures and `tmp_path`

```python
import pytest

@pytest.fixture
def report_dir(tmp_path):
    folder = tmp_path / "out"
    folder.mkdir()
    return folder

def test_first(report_dir):
    (report_dir / "report.json").write_text("{}", encoding="utf-8")
    assert len(list(report_dir.iterdir())) == 1

def test_second(report_dir):
    assert list(report_dir.iterdir()) == []
```

Does `test_second` pass after `test_first` has run? Why?

<details>
<summary>Show answer</summary>

**Yes.** pytest creates fixtures **per test** by default: each test gets a new `tmp_path` folder, so a new `report_dir`. Nothing `test_first` wrote is visible to `test_second`.

Also note that fixtures can use other fixtures (`report_dir` asks for `tmp_path`), and that pytest matches them purely **by parameter name**. A typo such as `def test_second(reportdir):` fails with `fixture 'reportdir' not found`, and lists the fixtures that are available.

</details>

## Q8 · Reading an error: the circular import

```text
Traceback (most recent call last):
  File "C:\work\incident-report\src\incident_report\cli.py", line 5, in <module>
    from incident_report.loading import load_config
  File "C:\work\incident-report\src\incident_report\loading.py", line 6, in <module>
    from incident_report.report import build_report
  File "C:\work\incident-report\src\incident_report\report.py", line 4, in <module>
    from incident_report.loading import FIELD_TYPES
ImportError: cannot import name 'FIELD_TYPES' from partially initialized module 'incident_report.loading' (most likely due to a circular import)
```

1. Which two modules form the cycle?
2. Why is `loading` "partially initialized"?
3. Propose a fix that respects the import graph in `01-concepts.ipynb`.

<details>
<summary>Show answer</summary>

1. `loading` and `report`: `loading` imports `report` (line 6), and `report` imports `loading` (line 4).
2. Python started executing `loading.py` and stopped at line 6 to import `report`. `report.py` then asked `loading` for `FIELD_TYPES`, but `loading` hasn't reached the line that defines it, so the name doesn't exist yet.
3. In the intended design, `loading` doesn't depend on `report` at all: `cli` coordinates them. Remove the `build_report` import from `loading`, and let `cli` call `load_exports` and then `build_report`. If `report` genuinely needs `FIELD_TYPES`, move that constant down into a module both can import, such as `models`. Dependencies should point one way, towards the modules with no internal imports.

</details>

## Q9 · Comparison

1. In Java, `private` fields can't be read from outside the class. What's the Python equivalent, and how is it enforced?
2. A JUnit 5 test class has `@BeforeEach` setup, `@Test` methods, and `@ParameterizedTest` with `@CsvSource`. Map each to pytest.
3. In C#, `public record Ticket(string Id, string Team);` gives value equality and immutability. What's the Python equivalent?

<details>
<summary>Show answer</summary>

1. A leading underscore: `self._checks`, `_type_names()`. It's a **convention**, not enforcement: anyone can still read it, but linters, editors and colleagues treat it as internal. (Two leading underscores trigger name mangling, which is rarely what you want.) The module equivalent is the same: `_helper()` means "not part of this module's public API".
2. `@BeforeEach` → a **fixture** that the test names as a parameter (and is created fresh per test). `@Test` → any function named `test_*`. `@ParameterizedTest` + `@CsvSource` → `@pytest.mark.parametrize("a, b", [(1, 2), ...])`.
3. `@dataclass(frozen=True) class Ticket: id: str; team: str`. It provides value-based `==`, a readable `repr`, immutability and hashing. Unlike C#'s compiler, nothing checks the field types at run time.

</details>

## Q10 · Engineering judgment: what to test

You have 48 provided acceptance tests for `incident_report`. Your workbook asks you to add your own unit tests. Which of these would you write, and which would you skip?

1. A test that `Ticket.from_record` turns an assignee of `"   "` into `None`.
2. A test that `deduplicate` calls `set.add` exactly once per unique ticket.
3. A test that `team_summary` reports `oldest_open_hours` as `None` for a team whose tickets are all closed.
4. A test that `load_exports` handles a file containing `[]`.
5. A test that `json.dumps` produces valid JSON.

<details>
<summary>Show answer</summary>

1. **Write it.** It's a business rule at an edge (blank versus missing), exactly where Sessions 02–03 found bugs.
2. **Skip it.** It tests *how* the function works, not *what* it returns. A refactor to `dict.fromkeys` would break the test without breaking anything users see.
3. **Write it**, if the acceptance tests don't already cover it. Empty groups and `max()` of nothing are classic edge cases. (Check first: one acceptance test covers this one, and duplicating it adds little.)
4. **Write it.** An empty export is valid and should produce no tickets and no rejections. It's an easy case to get wrong.
5. **Skip it.** That's testing the standard library. Test that *your* report contains only JSON-compatible types instead, for example with `json.loads(json.dumps(report)) == report`.

Good unit tests pin down **behaviour at the edges** of your own code, and are written so that refactoring doesn't break them.

</details>

## How did you do?

- **Q1, Q2, Q8** cover imports and running packages: Parts 1–2 of `01-concepts.ipynb`, and Labs 1–3.
- **Q3–Q5, Q9** cover classes, dataclasses and composition: Parts 3–4.
- **Q6, Q7, Q10** cover pytest: Part 5, and Labs 4–5.

Next: the workbook, which completes **Project 01**: [`../workbook/assignment.md`](../workbook/assignment.md).